# Missing-Value, Physical-Outlier, And Negative-Generation Audit

This notebook checks train datasets for missing values, physically impossible values, and negative generation values.

Outputs are saved under `outputs/missing_outlier_audit`.

Notes:
- `kpx_group_3` missing values through `2023-01-01 00:00:00` are expected and excluded from the audit missing summaries and event table.
- SCADA 10-minute timestamps are truncated to the hour before joining train labels.
- Conservative physical outlier rules target impossible values, not statistical extremes.
- For SCADA power, minor negative values are not treated as strict physical outliers; only values whose absolute magnitude exceeds the turbine-level upper bound are flagged.
- Negative generation values are audited separately for SCADA power and train-label generation.


In [1]:
from pathlib import Path
import re

import numpy as np
import pandas as pd

pd.set_option("display.max_columns", 180)
pd.set_option("display.width", 240)
pd.set_option("display.max_colwidth", 240)

BASE_DIR = Path.cwd()
TRAIN_DIR = BASE_DIR / "train"
OUTPUT_DIR = BASE_DIR / "outputs" / "missing_outlier_audit"
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

DATASET_PATHS = {
    "train_labels": TRAIN_DIR / "train_labels.csv",
    "ldaps_train": TRAIN_DIR / "ldaps_train.csv",
    "gfs_train": TRAIN_DIR / "gfs_train.csv",
    "scada_vestas_train": TRAIN_DIR / "scada_vestas_train.csv",
    "scada_unison_train": TRAIN_DIR / "scada_unison_train.csv",
}
TIME_COLUMNS = {
    "train_labels": "kst_dtm",
    "ldaps_train": "forecast_kst_dtm",
    "gfs_train": "forecast_kst_dtm",
    "scada_vestas_train": "kst_dtm",
    "scada_unison_train": "kst_dtm",
}
PARSE_DATE_COLUMNS = {
    "train_labels": ["kst_dtm"],
    "ldaps_train": ["forecast_kst_dtm", "data_available_kst_dtm"],
    "gfs_train": ["forecast_kst_dtm", "data_available_kst_dtm"],
    "scada_vestas_train": ["kst_dtm"],
    "scada_unison_train": ["kst_dtm"],
}
LABEL_COLS = ["kpx_group_1", "kpx_group_2", "kpx_group_3"]
LABEL_CAPACITY_KWH = {
    "kpx_group_1": 21600.0,
    "kpx_group_2": 21600.0,
    "kpx_group_3": 21000.0,
}
LABEL_CAPACITY_TOLERANCE = 1.05
EXPECTED_GROUP3_MISSING_THROUGH = pd.Timestamp("2023-01-01 00:00:00")
SCADA_POWER_ABS_UPPER = 5000.0
SCADA_WS_UPPER = 45.0
WIND_COMPONENT_ABS_UPPER = 100.0
PRESSURE_LOWER = 50000.0
PRESSURE_UPPER = 110000.0
TEMPERATURE_K_LOWER = 150.0
TEMPERATURE_K_UPPER = 350.0
RELATIVE_HUMIDITY_UPPER = 120.0
SPECIFIC_HUMIDITY_UPPER = 0.1
EPS = 1e-8

OUTPUT_PATHS = {
    "missing_dataset_summary": OUTPUT_DIR / "missing_dataset_summary.csv",
    "missing_column_summary": OUTPUT_DIR / "missing_column_summary.csv",
    "missing_events_detailed": OUTPUT_DIR / "missing_events_detailed.csv",
    "expected_missing_exclusions": OUTPUT_DIR / "expected_missing_exclusions.csv",
    "scada_missing_hourly_label_context": OUTPUT_DIR / "scada_missing_hourly_label_context.csv",
    "outlier_rule_config": OUTPUT_DIR / "physical_outlier_rule_config.csv",
    "outlier_dataset_summary": OUTPUT_DIR / "physical_outlier_dataset_summary.csv",
    "outlier_column_summary": OUTPUT_DIR / "physical_outlier_column_summary.csv",
    "outlier_events_detailed": OUTPUT_DIR / "physical_outlier_events_detailed.csv",
    "outlier_hourly_label_context": OUTPUT_DIR / "physical_outlier_hourly_label_context.csv",
    "scada_outlier_hourly_label_context": OUTPUT_DIR / "scada_physical_outlier_hourly_label_context.csv",
    "negative_generation_dataset_summary": OUTPUT_DIR / "negative_generation_dataset_summary.csv",
    "negative_generation_column_summary": OUTPUT_DIR / "negative_generation_column_summary.csv",
    "negative_generation_events_detailed": OUTPUT_DIR / "negative_generation_events_detailed.csv",
    "negative_generation_hourly_label_context": OUTPUT_DIR / "negative_generation_hourly_label_context.csv",
    "scada_negative_generation_hourly_label_context": OUTPUT_DIR / "scada_negative_generation_hourly_label_context.csv",
    "label_capacity_warning_events": OUTPUT_DIR / "label_capacity_warning_events.csv",
}

print("BASE_DIR:", BASE_DIR)
print("OUTPUT_DIR:", OUTPUT_DIR)

BASE_DIR: C:\Users\심현석\Documents\BARAM 2026
OUTPUT_DIR: C:\Users\심현석\Documents\BARAM 2026\outputs\missing_outlier_audit


## 1. Load Data

In [2]:
def read_dataset(dataset_name: str) -> pd.DataFrame:
    path = DATASET_PATHS[dataset_name]
    if not path.exists():
        raise FileNotFoundError(path)
    df = pd.read_csv(path, parse_dates=PARSE_DATE_COLUMNS[dataset_name])
    time_col = TIME_COLUMNS[dataset_name]
    df = df.sort_values(time_col).reset_index(drop=True)
    return df


datasets = {name: read_dataset(name) for name in DATASET_PATHS}
labels = datasets["train_labels"].copy()
label_context = labels[["kst_dtm"] + LABEL_COLS].rename(columns={"kst_dtm": "label_hour"})

for name, df in datasets.items():
    time_col = TIME_COLUMNS[name]
    print(name, df.shape, df[time_col].min(), "~", df[time_col].max())
    display(df.head(2))

train_labels (26304, 4) 2022-01-01 01:00:00 ~ 2025-01-01 00:00:00


,kst_dtm,kpx_group_1,kpx_group_2,kpx_group_3
0,2022-01-01 01:00:00,12004.421,9719.242,NaN
1,2022-01-01 02:00:00,12901.137,10297.768,NaN


ldaps_train (420864, 35) 2022-01-01 01:00:00 ~ 2025-01-01 00:00:00


,forecast_kst_dtm,data_available_kst_dtm,grid_id,latitude,longitude,heightAboveGround_10_10u,heightAboveGround_10_10v,heightAboveGround_50_50MUmax,heightAboveGround_50_50MUmin,heightAboveGround_50_50MVmax,heightAboveGround_50_50MVmin,heightAboveGround_5_XBLWS,heightAboveGround_5_YBLWS,heightAboveGround_2_t,heightAboveGround_2_dpt,heightAboveGround_2_r,heightAboveGround_2_q,surface_0_sp,meanSea_0_prmsl,etc_0_blh,surface_0_NDNSW,surface_0_NDNLW,heightAboveGround_2_SWDIR,heightAboveGround_2_SWDIF,etc_0_hcc,etc_0_mcc,etc_0_lcc,etc_0_VLCDC,surface_0_avg_lsprate,surface_0_lssrate,surface_0_ncpcp,surface_0_snol,surface_0_SNOM,surface_0_lsm,surface_0_h
0,2022-01-01 01:00:00,2021-12-31 13:00:00,1,37.3032,128.9443,5.411171,1.591679,8.162608,7.618145,2.164301,1.766520,0.013109,0.191541,260.74554,246.23834,32.642956,0.000488,90628.91,102664.920,177.19792,0.0,-95.399080,0.0,0.0,0.093750,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0,992.625
1,2022-01-01 01:00:00,2021-12-31 13:00:00,16,37.2607,128.9771,6.571816,-1.215450,10.459971,9.327129,-1.499762,-1.684652,0.009447,-0.212023,261.40472,246.00885,30.174206,0.000488,91289.91,102638.266,164.63542,0.0,-96.590485,0.0,0.0,0.109375,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0,933.750


gfs_train (236736, 40) 2022-01-01 01:00:00 ~ 2025-01-01 00:00:00


,forecast_kst_dtm,data_available_kst_dtm,grid_id,latitude,longitude,heightAboveGround_10_10u,heightAboveGround_10_10v,heightAboveGround_80_u,heightAboveGround_80_v,heightAboveGround_100_100u,heightAboveGround_100_100v,heightAboveGround_2_2t,heightAboveGround_2_2d,heightAboveGround_2_2r,heightAboveGround_2_2sh,planetaryBoundaryLayer_0_u,planetaryBoundaryLayer_0_v,planetaryBoundaryLayer_0_VRATE,surface_0_dswrf,surface_0_dlwrf,surface_0_prate,surface_0_tp,surface_0_sp,meanSea_0_prmsl,surface_0_gust,lowCloudLayer_0_lcc,middleCloudLayer_0_mcc,highCloudLayer_0_hcc,atmosphere_0_tcc,isobaricInhPa_850_t,isobaricInhPa_850_u,isobaricInhPa_850_v,isobaricInhPa_850_r,isobaricInhPa_700_t,isobaricInhPa_700_u,isobaricInhPa_700_v,isobaricInhPa_500_gh,isobaricInhPa_500_t,isobaricInhPa_500_u,isobaricInhPa_500_v
0,2022-01-01 01:00:00,2021-12-31 13:00:00,1,37.5,128.75,1.817466,1.040068,2.437944,1.258201,2.539397,1.267655,260.3425,249.06181,41.5,0.000583,1.968707,1.032028,0.0,0.0,162.56856,0.0,0.0,93651.410,103163.5,2.601783,0.0,0.0,4.8,5.0,265.62122,5.928218,-3.632051,8.5,260.32092,11.584992,-18.589058,5474.115,243.86014,22.805730,-15.617361
1,2022-01-01 01:00:00,2021-12-31 13:00:00,2,37.5,129.00,2.947466,1.340068,3.747944,1.728201,3.839397,1.797655,262.5225,250.18182,37.7,0.000621,3.168707,1.432028,0.0,0.0,168.66855,0.0,0.0,97209.805,103011.1,4.101783,0.0,0.0,4.6,4.6,265.56122,4.158218,-2.582051,8.3,259.80090,11.744992,-18.769058,5470.083,244.00014,23.905731,-16.017360


scada_vestas_train (157819, 37) 2022-01-01 01:00:00 ~ 2025-01-01 00:00:00


,kst_dtm,vestas_wtg01_power_kw10m,vestas_wtg02_power_kw10m,vestas_wtg03_power_kw10m,vestas_wtg04_power_kw10m,vestas_wtg05_power_kw10m,vestas_wtg06_power_kw10m,vestas_wtg07_power_kw10m,vestas_wtg08_power_kw10m,vestas_wtg09_power_kw10m,vestas_wtg10_power_kw10m,vestas_wtg11_power_kw10m,vestas_wtg12_power_kw10m,vestas_wtg01_ws,vestas_wtg02_ws,vestas_wtg03_ws,vestas_wtg04_ws,vestas_wtg05_ws,vestas_wtg06_ws,vestas_wtg07_ws,vestas_wtg08_ws,vestas_wtg09_ws,vestas_wtg10_ws,vestas_wtg11_ws,vestas_wtg12_ws,vestas_wtg01_wd,vestas_wtg02_wd,vestas_wtg03_wd,vestas_wtg04_wd,vestas_wtg05_wd,vestas_wtg06_wd,vestas_wtg07_wd,vestas_wtg08_wd,vestas_wtg09_wd,vestas_wtg10_wd,vestas_wtg11_wd,vestas_wtg12_wd
0,2022-01-01 01:00:00,325,282,383,470,423,122,200,206,195,326,364,245,8.5,8.2,8.9,9.5,8.8,5.5,7.8,7.7,7.4,8.0,8.5,7.4,314.8,259.7,281.2,308.0,275.6,285.9,270.7,278.1,292.6,247.2,291.0,95.2
1,2022-01-01 01:10:00,338,312,367,420,370,87,153,215,234,412,426,291,8.8,8.5,9.0,9.2,8.3,5.1,6.8,7.4,7.8,8.7,9.1,7.6,315.5,260.4,283.8,307.9,273.8,290.5,268.7,283.1,301.7,253.2,295.3,96.2


scada_unison_train (105264, 16) 2023-01-01 00:10:00 ~ 2025-01-01 00:00:00


,kst_dtm,unison_wtg01_power_kw10m,unison_wtg02_power_kw10m,unison_wtg03_power_kw10m,unison_wtg04_power_kw10m,unison_wtg05_power_kw10m,unison_wtg01_ws,unison_wtg02_ws,unison_wtg03_ws,unison_wtg04_ws,unison_wtg05_ws,unison_wtg01_wd,unison_wtg02_wd,unison_wtg03_wd,unison_wtg04_wd,unison_wtg05_wd
0,2023-01-01 00:10:00,706.0,702.0,0.0,705.0,707.0,13.73,15.03,16.42,13.97,14.57,-79.005327,-69.946609,-124.739623,-90.938062,-130.941880
1,2023-01-01 00:20:00,706.0,702.0,0.0,684.0,708.0,13.61,14.76,16.51,14.96,14.34,-77.479172,-68.192593,-122.671765,-87.976756,-131.394959


## 2. Missing-Value Audit

In [3]:
def dataset_source(dataset_name: str) -> str:
    if dataset_name == "ldaps_train":
        return "ldaps"
    if dataset_name == "gfs_train":
        return "gfs"
    if dataset_name == "scada_vestas_train":
        return "vestas"
    if dataset_name == "scada_unison_train":
        return "unison"
    if dataset_name == "train_labels":
        return "label"
    return dataset_name


def label_join_hour(dataset_name: str, event_time: pd.Series) -> pd.Series:
    event_time = pd.to_datetime(event_time)
    return event_time.dt.floor("h")


def unique_join(values: pd.Series) -> str:
    out = []
    seen = set()
    for item in values.dropna().astype(str):
        for token in item.split("|"):
            token = token.strip()
            if token and token not in seen:
                out.append(token)
                seen.add(token)
    return "|".join(out)


def expected_missing_mask(dataset_name: str, df: pd.DataFrame, column: str) -> pd.Series:
    mask = pd.Series(False, index=df.index)
    if dataset_name == "train_labels" and column == "kpx_group_3":
        mask = pd.to_datetime(df["kst_dtm"]) <= EXPECTED_GROUP3_MISSING_THROUGH
    return mask


def audit_missing_matrix(dataset_name: str, df: pd.DataFrame) -> pd.DataFrame:
    missing = df.isna().copy()
    for col in df.columns:
        expected = missing[col] & expected_missing_mask(dataset_name, df, col)
        if expected.any():
            missing.loc[expected, col] = False
    return missing


def summarize_expected_missing_exclusions(dataset_name: str, df: pd.DataFrame) -> pd.DataFrame:
    rows = []
    raw_missing = df.isna()
    for col in df.columns:
        expected = raw_missing[col] & expected_missing_mask(dataset_name, df, col)
        if expected.any():
            time_col = TIME_COLUMNS[dataset_name]
            event_times = pd.to_datetime(df.loc[expected, time_col])
            rows.append({
                "dataset": dataset_name,
                "source": dataset_source(dataset_name),
                "column": col,
                "excluded_missing_count": int(expected.sum()),
                "first_excluded_time": event_times.min(),
                "last_excluded_time": event_times.max(),
                "reason": f"Expected missing values through {EXPECTED_GROUP3_MISSING_THROUGH}",
            })
    return pd.DataFrame(rows)


def summarize_missing_dataset(dataset_name: str, df: pd.DataFrame) -> dict:
    missing = audit_missing_matrix(dataset_name, df)
    return {
        "dataset": dataset_name,
        "source": dataset_source(dataset_name),
        "n_rows": int(len(df)),
        "n_columns": int(df.shape[1]),
        "n_missing_cells": int(missing.sum().sum()),
        "n_rows_with_missing": int(missing.any(axis=1).sum()),
        "n_columns_with_missing": int((missing.sum(axis=0) > 0).sum()),
        "missing_cell_rate": float(missing.sum().sum() / (df.shape[0] * df.shape[1])),
    }


def summarize_missing_columns(dataset_name: str, df: pd.DataFrame) -> pd.DataFrame:
    missing = audit_missing_matrix(dataset_name, df)
    missing_counts = missing.sum()
    out = pd.DataFrame({
        "dataset": dataset_name,
        "source": dataset_source(dataset_name),
        "column": missing_counts.index,
        "missing_count": missing_counts.values.astype(int),
        "missing_rate": missing_counts.values / len(df),
        "dtype": [str(df[col].dtype) for col in missing_counts.index],
    })
    return out.sort_values(["missing_count", "column"], ascending=[False, True]).reset_index(drop=True)


def missing_events_for_dataset(dataset_name: str, df: pd.DataFrame) -> pd.DataFrame:
    missing = audit_missing_matrix(dataset_name, df)
    cols = [col for col in df.columns if missing[col].any()]
    if not cols:
        return pd.DataFrame()
    time_col = TIME_COLUMNS[dataset_name]
    context_cols = []
    if "grid_id" in df.columns:
        context_cols.append("grid_id")
    event_frames = []
    for col in cols:
        mask = missing[col]
        frame_cols = [time_col] + context_cols
        frame = df.loc[mask, frame_cols].copy()
        frame["dataset"] = dataset_name
        frame["source"] = dataset_source(dataset_name)
        frame["event_time"] = frame[time_col]
        frame["label_hour"] = label_join_hour(dataset_name, frame["event_time"])
        frame["column"] = col
        frame["event_type"] = "missing"
        frame["value"] = np.nan
        if dataset_name.startswith("scada_"):
            frame["turbine_id"] = frame["column"].str.extract(r"(wtg\d+)", expand=False)
            frame["variable_type"] = frame["column"].str.extract(r"_(power_kw10m|ws|wd)$", expand=False)
        else:
            frame["turbine_id"] = np.nan
            frame["variable_type"] = col
        event_frames.append(frame.drop(columns=[time_col]))
    events = pd.concat(event_frames, ignore_index=True) if event_frames else pd.DataFrame()
    return events


missing_dataset_summary = pd.DataFrame([summarize_missing_dataset(name, df) for name, df in datasets.items()])
missing_column_summary = pd.concat([summarize_missing_columns(name, df) for name, df in datasets.items()], ignore_index=True)
missing_events_detailed = pd.concat([missing_events_for_dataset(name, df) for name, df in datasets.items()], ignore_index=True)
expected_missing_exclusions = pd.concat(
    [summarize_expected_missing_exclusions(name, df) for name, df in datasets.items()],
    ignore_index=True,
)
if not missing_events_detailed.empty:
    missing_events_detailed = missing_events_detailed.merge(label_context, on="label_hour", how="left")

missing_dataset_summary.to_csv(OUTPUT_PATHS["missing_dataset_summary"], index=False, encoding="utf-8-sig")
missing_column_summary.to_csv(OUTPUT_PATHS["missing_column_summary"], index=False, encoding="utf-8-sig")
missing_events_detailed.to_csv(OUTPUT_PATHS["missing_events_detailed"], index=False, encoding="utf-8-sig")
expected_missing_exclusions.to_csv(OUTPUT_PATHS["expected_missing_exclusions"], index=False, encoding="utf-8-sig")

print("Expected missing exclusions:")
display(expected_missing_exclusions)
print("Missing dataset summary, after expected exclusions:")
display(missing_dataset_summary)
print("Columns with missing values, after expected exclusions:")
display(missing_column_summary[missing_column_summary["missing_count"] > 0])
print("Missing events detailed shape:", missing_events_detailed.shape)
display(missing_events_detailed.head(20))


Expected missing exclusions:


,dataset,source,column,excluded_missing_count,first_excluded_time,last_excluded_time,reason
0,train_labels,label,kpx_group_3,8760,2022-01-01 01:00:00,2023-01-01,Expected missing values through 2023-01-01 00:00:00


Missing dataset summary, after expected exclusions:


,dataset,source,n_rows,n_columns,n_missing_cells,n_rows_with_missing,n_columns_with_missing,missing_cell_rate
0,train_labels,label,26304,4,213,106,3,0.002024
1,ldaps_train,ldaps,420864,35,0,0,0,0.000000
2,gfs_train,gfs,236736,40,0,0,0,0.000000
3,scada_vestas_train,vestas,157819,37,0,0,0,0.000000
4,scada_unison_train,unison,105264,16,9511,4734,15,0.005647


Columns with missing values, after expected exclusions:


,dataset,source,column,missing_count,missing_rate,dtype
0,train_labels,label,kpx_group_1,104,0.003954,float64
1,train_labels,label,kpx_group_2,103,0.003916,float64
2,train_labels,label,kpx_group_3,6,0.000228,float64
116,scada_unison_train,unison,unison_wtg01_ws,1466,0.013927,float64
117,scada_unison_train,unison,unison_wtg02_ws,1424,0.013528,float64
118,scada_unison_train,unison,unison_wtg05_ws,1379,0.013100,float64
119,scada_unison_train,unison,unison_wtg02_power_kw10m,1362,0.012939,float64
120,scada_unison_train,unison,unison_wtg02_wd,1354,0.012863,float64
121,scada_unison_train,unison,unison_wtg03_ws,447,0.004246,float64
122,scada_unison_train,unison,unison_wtg04_ws,380,0.003610,float64


Missing events detailed shape: (9724, 12)


,dataset,source,event_time,label_hour,column,event_type,value,turbine_id,variable_type,kpx_group_1,kpx_group_2,kpx_group_3
0,train_labels,label,2022-03-04 14:00:00,2022-03-04 14:00:00,kpx_group_1,missing,NaN,NaN,kpx_group_1,NaN,NaN,NaN
1,train_labels,label,2022-03-04 15:00:00,2022-03-04 15:00:00,kpx_group_1,missing,NaN,NaN,kpx_group_1,NaN,NaN,NaN
2,train_labels,label,2022-07-04 09:00:00,2022-07-04 09:00:00,kpx_group_1,missing,NaN,NaN,kpx_group_1,NaN,NaN,NaN
3,train_labels,label,2022-07-04 10:00:00,2022-07-04 10:00:00,kpx_group_1,missing,NaN,NaN,kpx_group_1,NaN,NaN,NaN
4,train_labels,label,2022-07-04 11:00:00,2022-07-04 11:00:00,kpx_group_1,missing,NaN,NaN,kpx_group_1,NaN,NaN,NaN
5,train_labels,label,2022-07-04 12:00:00,2022-07-04 12:00:00,kpx_group_1,missing,NaN,NaN,kpx_group_1,NaN,NaN,NaN
6,train_labels,label,2022-07-04 13:00:00,2022-07-04 13:00:00,kpx_group_1,missing,NaN,NaN,kpx_group_1,NaN,NaN,NaN
7,train_labels,label,2022-07-04 14:00:00,2022-07-04 14:00:00,kpx_group_1,missing,NaN,NaN,kpx_group_1,NaN,NaN,NaN
8,train_labels,label,2022-07-04 15:00:00,2022-07-04 15:00:00,kpx_group_1,missing,NaN,NaN,kpx_group_1,NaN,NaN,NaN
9,train_labels,label,2022-07-04 16:00:00,2022-07-04 16:00:00,kpx_group_1,missing,NaN,NaN,kpx_group_1,NaN,NaN,NaN


## 3. SCADA Missing Events With Train-Label Context

In [4]:
def make_hourly_missing_context(events: pd.DataFrame, scada_only: bool = True) -> pd.DataFrame:
    if events.empty:
        return pd.DataFrame()
    data = events.copy()
    if scada_only:
        data = data[data["dataset"].str.startswith("scada_")].copy()
    if data.empty:
        return pd.DataFrame()
    context = (
        data.groupby(["dataset", "source", "label_hour"], as_index=False)
        .agg(
            n_event_rows=("event_time", "nunique"),
            n_missing_cells=("column", "size"),
            missing_columns=("column", unique_join),
            missing_turbines=("turbine_id", unique_join),
            missing_variable_types=("variable_type", unique_join),
        )
        .merge(label_context, on="label_hour", how="left")
        .sort_values(["dataset", "label_hour"])
    )
    return context


scada_missing_hourly_label_context = make_hourly_missing_context(missing_events_detailed, scada_only=True)
scada_missing_hourly_label_context.to_csv(OUTPUT_PATHS["scada_missing_hourly_label_context"], index=False, encoding="utf-8-sig")

print("SCADA missing hourly label context:", scada_missing_hourly_label_context.shape)
display(scada_missing_hourly_label_context.head(30))
print("SCADA missing hourly context by source:")
if not scada_missing_hourly_label_context.empty:
    display(scada_missing_hourly_label_context.groupby("source").agg(
        n_hours=("label_hour", "nunique"),
        n_missing_cells=("n_missing_cells", "sum"),
        first_hour=("label_hour", "min"),
        last_hour=("label_hour", "max"),
    ).reset_index())

SCADA missing hourly label context: (961, 11)


,dataset,source,label_hour,n_event_rows,n_missing_cells,missing_columns,missing_turbines,missing_variable_types,kpx_group_1,kpx_group_2,kpx_group_3
0,scada_unison_train,unison,2023-01-06 09:00:00,3,3,unison_wtg04_ws,wtg04,ws,9343.200,12973.453,7217.116
1,scada_unison_train,unison,2023-01-11 09:00:00,1,1,unison_wtg05_ws,wtg05,ws,19843.453,20754.632,14159.432
2,scada_unison_train,unison,2023-01-12 09:00:00,1,1,unison_wtg05_ws,wtg05,ws,17095.453,12351.537,3413.305
3,scada_unison_train,unison,2023-01-13 09:00:00,2,2,unison_wtg03_ws|unison_wtg05_ws,wtg03|wtg05,ws,10095.284,10601.495,4454.653
4,scada_unison_train,unison,2023-01-13 10:00:00,5,6,unison_wtg03_ws|unison_wtg05_ws,wtg03|wtg05,ws,11686.232,9863.874,3688.105
5,scada_unison_train,unison,2023-01-13 11:00:00,3,3,unison_wtg05_ws,wtg05,ws,11020.926,12438.316,5293.516
6,scada_unison_train,unison,2023-01-18 10:00:00,4,12,unison_wtg05_power_kw10m|unison_wtg05_ws|unison_wtg05_wd,wtg05,power_kw10m|ws|wd,1836.821,2256.253,592.989
7,scada_unison_train,unison,2023-01-18 11:00:00,6,18,unison_wtg05_power_kw10m|unison_wtg05_ws|unison_wtg05_wd,wtg05,power_kw10m|ws|wd,882.253,1084.737,216.947
8,scada_unison_train,unison,2023-01-18 12:00:00,1,2,unison_wtg05_power_kw10m|unison_wtg05_ws,wtg05,power_kw10m|ws,650.842,433.895,86.779
9,scada_unison_train,unison,2023-01-18 16:00:00,1,1,unison_wtg04_ws,wtg04,ws,7231.579,3601.326,2863.705


SCADA missing hourly context by source:


,source,n_hours,n_missing_cells,first_hour,last_hour
0,unison,961,9511,2023-01-06 09:00:00,2025-01-01


## 4. Physical-Outlier Rules

The rules below are conservative physical-validity checks. They are not intended to flag all statistically unusual values.

In [5]:
def variable_type_from_column(dataset_name: str, column: str) -> str:
    if dataset_name.startswith("scada_"):
        if "_power_" in column:
            return "power"
        if column.endswith("_ws"):
            return "wind_speed"
        if column.endswith("_wd"):
            return "wind_direction"
    if column in LABEL_COLS:
        return "generation"
    if column in ["surface_0_sp", "meanSea_0_prmsl"]:
        return "pressure"
    if re.search(r"(_t$|_2t$|_dpt$|_2d$|isobaricInhPa_\d+_t$)", column):
        return "temperature"
    if re.search(r"(_r$|_2r$|isobaricInhPa_\d+_r$)", column):
        return "relative_humidity"
    if column.endswith("_q") or column.endswith("_2sh"):
        return "specific_humidity"
    if any(token in column for token in ["hcc", "mcc", "lcc", "VLCDC", "tcc"]):
        return "cloud_cover"
    if any(token in column for token in ["prate", "tp", "lsprate", "lssrate", "ncpcp", "snol", "SNOM"]):
        return "precip_or_snow"
    if any(token in column for token in ["SWDIR", "SWDIF", "dswrf", "dlwrf", "NDNSW"]):
        return "radiation_nonnegative"
    if column.endswith("_gust"):
        return "wind_speed"
    if re.search(r"(_u$|_v$|10u|10v|80_u|80_v|100u|100v|MUmax|MUmin|MVmax|MVmin|XBLWS|YBLWS)", column):
        return "wind_component"
    if column in ["latitude", "longitude", "grid_id"]:
        return column
    return "other"


def rules_for_column(dataset_name: str, source: str, column: str):
    rules = []
    vtype = variable_type_from_column(dataset_name, column)
    if column in LABEL_COLS:
        capacity = LABEL_CAPACITY_KWH[column]
        rules.append(("generation_lt_0", lambda s, cap=capacity: s.notna() & (s < 0)))
        rules.append((f"generation_gt_{LABEL_CAPACITY_TOLERANCE:.2f}x_capacity", lambda s, cap=capacity: s.notna() & (s > cap * LABEL_CAPACITY_TOLERANCE)))
        return vtype, rules

    if dataset_name.startswith("scada_"):
        if vtype == "power":
            rules.append((
                f"scada_power_abs_gt_{SCADA_POWER_ABS_UPPER:g}",
                lambda s: s.notna() & (s.abs() > SCADA_POWER_ABS_UPPER),
            ))
        elif vtype == "wind_speed":
            rules.append(("scada_ws_lt_0", lambda s: s.notna() & (s < 0)))
            rules.append((f"scada_ws_gt_{SCADA_WS_UPPER:g}", lambda s: s.notna() & (s > SCADA_WS_UPPER)))
        elif vtype == "wind_direction":
            if source == "vestas":
                rules.append(("vestas_wd_outside_0_360", lambda s: s.notna() & ((s < 0) | (s >= 360))))
            elif source == "unison":
                rules.append(("unison_wd_outside_minus180_180", lambda s: s.notna() & ((s < -180) | (s > 180))))
        return vtype, rules

    if dataset_name in ["ldaps_train", "gfs_train"]:
        if column == "grid_id":
            rules.append(("grid_id_nonpositive", lambda s: s.notna() & (s <= 0)))
        elif column == "latitude":
            rules.append(("latitude_outside_minus90_90", lambda s: s.notna() & ((s < -90) | (s > 90))))
        elif column == "longitude":
            rules.append(("longitude_outside_minus180_180", lambda s: s.notna() & ((s < -180) | (s > 180))))
        elif vtype == "pressure":
            rules.append(("pressure_pa_outside_50000_110000", lambda s: s.notna() & ((s < PRESSURE_LOWER) | (s > PRESSURE_UPPER))))
        elif vtype == "temperature":
            rules.append(("temperature_k_outside_150_350", lambda s: s.notna() & ((s < TEMPERATURE_K_LOWER) | (s > TEMPERATURE_K_UPPER))))
        elif vtype == "relative_humidity":
            rules.append((f"relative_humidity_outside_0_{RELATIVE_HUMIDITY_UPPER:g}", lambda s: s.notna() & ((s < -EPS) | (s > RELATIVE_HUMIDITY_UPPER))))
        elif vtype == "specific_humidity":
            rules.append((f"specific_humidity_outside_0_{SPECIFIC_HUMIDITY_UPPER:g}", lambda s: s.notna() & ((s < -EPS) | (s > SPECIFIC_HUMIDITY_UPPER))))
        elif vtype == "cloud_cover":
            if dataset_name == "ldaps_train":
                rules.append(("ldaps_cloud_fraction_outside_0_1.05", lambda s: s.notna() & ((s < -EPS) | (s > 1.05))))
            else:
                rules.append(("gfs_cloud_percent_outside_0_100", lambda s: s.notna() & ((s < -EPS) | (s > 100))))
        elif vtype in ["precip_or_snow", "radiation_nonnegative"]:
            rules.append((f"{vtype}_lt_0", lambda s: s.notna() & (s < -EPS)))
        elif vtype == "wind_speed":
            rules.append(("weather_wind_speed_lt_0", lambda s: s.notna() & (s < -EPS)))
            rules.append(("weather_wind_speed_gt_100", lambda s: s.notna() & (s > 100)))
        elif vtype == "wind_component":
            rules.append((f"wind_component_abs_gt_{WIND_COMPONENT_ABS_UPPER:g}", lambda s: s.notna() & (s.abs() > WIND_COMPONENT_ABS_UPPER)))
    return vtype, rules


rule_rows = []
for dataset_name, df in datasets.items():
    source = dataset_source(dataset_name)
    for col in df.select_dtypes(include=[np.number]).columns:
        vtype, rules = rules_for_column(dataset_name, source, col)
        for rule_name, _ in rules:
            rule_rows.append({
                "dataset": dataset_name,
                "source": source,
                "column": col,
                "variable_type": vtype,
                "rule": rule_name,
            })

outlier_rule_config = pd.DataFrame(rule_rows)
outlier_rule_config.to_csv(OUTPUT_PATHS["outlier_rule_config"], index=False, encoding="utf-8-sig")
print("Configured physical outlier rules:", outlier_rule_config.shape)
display(outlier_rule_config)

Configured physical outlier rules: (140, 5)


,dataset,source,column,variable_type,rule
0,train_labels,label,kpx_group_1,generation,generation_lt_0
1,train_labels,label,kpx_group_1,generation,generation_gt_1.05x_capacity
2,train_labels,label,kpx_group_2,generation,generation_lt_0
3,train_labels,label,kpx_group_2,generation,generation_gt_1.05x_capacity
4,train_labels,label,kpx_group_3,generation,generation_lt_0
...,...,...,...,...,...
135,scada_unison_train,unison,unison_wtg01_wd,wind_direction,unison_wd_outside_minus180_180
136,scada_unison_train,unison,unison_wtg02_wd,wind_direction,unison_wd_outside_minus180_180
137,scada_unison_train,unison,unison_wtg03_wd,wind_direction,unison_wd_outside_minus180_180
138,scada_unison_train,unison,unison_wtg04_wd,wind_direction,unison_wd_outside_minus180_180


## 5. Physical-Outlier Audit

In [6]:
def turbine_id_from_scada_column(column: str) -> str | float:
    match = re.search(r"(wtg\d+)", column)
    return match.group(1) if match else np.nan


def outlier_events_for_dataset(dataset_name: str, df: pd.DataFrame) -> pd.DataFrame:
    source = dataset_source(dataset_name)
    time_col = TIME_COLUMNS[dataset_name]
    context_cols = []
    for col in ["grid_id", "latitude", "longitude", "data_available_kst_dtm"]:
        if col in df.columns:
            context_cols.append(col)
    event_frames = []
    numeric_cols = df.select_dtypes(include=[np.number]).columns.tolist()
    for col in numeric_cols:
        vtype, rules = rules_for_column(dataset_name, source, col)
        if not rules:
            continue
        values = df[col]
        for rule_name, mask_func in rules:
            mask = mask_func(values)
            n_hit = int(mask.sum())
            if n_hit == 0:
                continue
            frame_cols = [time_col] + context_cols + [col]
            frame = df.loc[mask, frame_cols].copy()
            frame["dataset"] = dataset_name
            frame["source"] = source
            frame["event_time"] = frame[time_col]
            frame["label_hour"] = label_join_hour(dataset_name, frame["event_time"])
            frame["column"] = col
            frame["variable_type"] = vtype
            frame["rule"] = rule_name
            frame["value"] = frame[col]
            if dataset_name.startswith("scada_"):
                frame["turbine_id"] = turbine_id_from_scada_column(col)
            else:
                frame["turbine_id"] = np.nan
            drop_cols = [time_col, col]
            frame = frame.drop(columns=drop_cols)
            event_frames.append(frame)
    if not event_frames:
        return pd.DataFrame()
    return pd.concat(event_frames, ignore_index=True)


outlier_events_list = []
for name, df in datasets.items():
    events = outlier_events_for_dataset(name, df)
    print(name, events.shape)
    outlier_events_list.append(events)

outlier_events_detailed = pd.concat(outlier_events_list, ignore_index=True) if outlier_events_list else pd.DataFrame()
if not outlier_events_detailed.empty:
    outlier_events_detailed = outlier_events_detailed.merge(label_context, on="label_hour", how="left")

if outlier_events_detailed.empty:
    outlier_column_summary = pd.DataFrame(columns=["dataset", "source", "column", "variable_type", "rule", "outlier_count", "first_event_time", "last_event_time", "min_value", "max_value"])
    outlier_dataset_summary = pd.DataFrame(columns=["dataset", "source", "n_outlier_cells", "n_event_hours", "n_columns_with_outliers", "first_event_time", "last_event_time"])
else:
    outlier_column_summary = (
        outlier_events_detailed.groupby(["dataset", "source", "column", "variable_type", "rule"], as_index=False)
        .agg(
            outlier_count=("value", "size"),
            first_event_time=("event_time", "min"),
            last_event_time=("event_time", "max"),
            min_value=("value", "min"),
            max_value=("value", "max"),
        )
        .sort_values(["outlier_count", "dataset", "column"], ascending=[False, True, True])
    )
    outlier_dataset_summary = (
        outlier_events_detailed.groupby(["dataset", "source"], as_index=False)
        .agg(
            n_outlier_cells=("value", "size"),
            n_event_hours=("label_hour", "nunique"),
            n_columns_with_outliers=("column", "nunique"),
            first_event_time=("event_time", "min"),
            last_event_time=("event_time", "max"),
        )
        .sort_values("n_outlier_cells", ascending=False)
    )

outlier_dataset_summary.to_csv(OUTPUT_PATHS["outlier_dataset_summary"], index=False, encoding="utf-8-sig")
outlier_column_summary.to_csv(OUTPUT_PATHS["outlier_column_summary"], index=False, encoding="utf-8-sig")
outlier_events_detailed.to_csv(OUTPUT_PATHS["outlier_events_detailed"], index=False, encoding="utf-8-sig")

print("Physical outlier dataset summary:")
display(outlier_dataset_summary)
print("Physical outlier column summary:")
display(outlier_column_summary.head(60))
print("Physical outlier detailed events shape:", outlier_events_detailed.shape)
display(outlier_events_detailed.head(20))

train_labels (0, 0)
ldaps_train (0, 0)


gfs_train (0, 0)


scada_vestas_train (868, 9)
scada_unison_train (0, 0)
Physical outlier dataset summary:


,dataset,source,n_outlier_cells,n_event_hours,n_columns_with_outliers,first_event_time,last_event_time
0,scada_vestas_train,vestas,868,455,12,2022-01-11 20:20:00,2024-12-28 11:00:00


Physical outlier column summary:


,dataset,source,column,variable_type,rule,outlier_count,first_event_time,last_event_time,min_value,max_value
7,scada_vestas_train,vestas,vestas_wtg08_power_kw10m,power,scada_power_abs_gt_5000,102,2022-03-04 14:20:00,2024-12-23 19:20:00,-42301001,42300996
10,scada_vestas_train,vestas,vestas_wtg11_power_kw10m,power,scada_power_abs_gt_5000,98,2022-03-04 14:20:00,2024-12-28 11:00:00,-51770425,51770425
2,scada_vestas_train,vestas,vestas_wtg03_power_kw10m,power,scada_power_abs_gt_5000,88,2022-03-04 14:20:00,2024-11-20 14:00:00,-46559445,46559445
0,scada_vestas_train,vestas,vestas_wtg01_power_kw10m,power,scada_power_abs_gt_5000,74,2022-02-21 15:40:00,2024-12-05 18:30:00,-42512957,42512957
4,scada_vestas_train,vestas,vestas_wtg05_power_kw10m,power,scada_power_abs_gt_5000,72,2022-01-30 19:10:00,2024-10-15 13:30:00,-33717814,33717814
11,scada_vestas_train,vestas,vestas_wtg12_power_kw10m,power,scada_power_abs_gt_5000,72,2022-03-04 14:20:00,2024-10-14 17:40:00,-41447795,41447795
1,scada_vestas_train,vestas,vestas_wtg02_power_kw10m,power,scada_power_abs_gt_5000,70,2022-01-13 10:30:00,2024-10-14 14:20:00,-41959813,41959813
6,scada_vestas_train,vestas,vestas_wtg07_power_kw10m,power,scada_power_abs_gt_5000,68,2022-02-04 18:40:00,2024-11-29 19:00:00,-29770834,29770834
9,scada_vestas_train,vestas,vestas_wtg10_power_kw10m,power,scada_power_abs_gt_5000,68,2022-03-14 18:20:00,2024-10-10 15:40:00,-51376454,51376454
5,scada_vestas_train,vestas,vestas_wtg06_power_kw10m,power,scada_power_abs_gt_5000,60,2022-01-11 20:20:00,2024-10-15 16:20:00,-34184409,34184409


Physical outlier detailed events shape: (868, 12)


,dataset,source,event_time,label_hour,column,variable_type,rule,value,turbine_id,kpx_group_1,kpx_group_2,kpx_group_3
0,scada_vestas_train,vestas,2022-02-21 15:40:00,2022-02-21 15:00:00,vestas_wtg01_power_kw10m,power,scada_power_abs_gt_5000,-15326974,wtg01,12149.053,14087.116,NaN
1,scada_vestas_train,vestas,2022-02-21 15:50:00,2022-02-21 15:00:00,vestas_wtg01_power_kw10m,power,scada_power_abs_gt_5000,15326971,wtg01,12149.053,14087.116,NaN
2,scada_vestas_train,vestas,2022-03-04 14:00:00,2022-03-04 14:00:00,vestas_wtg01_power_kw10m,power,scada_power_abs_gt_5000,-16006532,wtg01,NaN,NaN,NaN
3,scada_vestas_train,vestas,2022-03-04 15:30:00,2022-03-04 15:00:00,vestas_wtg01_power_kw10m,power,scada_power_abs_gt_5000,16006654,wtg01,NaN,NaN,NaN
4,scada_vestas_train,vestas,2022-03-04 15:40:00,2022-03-04 15:00:00,vestas_wtg01_power_kw10m,power,scada_power_abs_gt_5000,-16006654,wtg01,NaN,NaN,NaN
5,scada_vestas_train,vestas,2022-03-04 18:40:00,2022-03-04 18:00:00,vestas_wtg01_power_kw10m,power,scada_power_abs_gt_5000,16006652,wtg01,4975.326,16473.537,NaN
6,scada_vestas_train,vestas,2022-07-04 08:50:00,2022-07-04 08:00:00,vestas_wtg01_power_kw10m,power,scada_power_abs_gt_5000,-19728552,wtg01,0.000,0.000,NaN
7,scada_vestas_train,vestas,2022-07-04 19:20:00,2022-07-04 19:00:00,vestas_wtg01_power_kw10m,power,scada_power_abs_gt_5000,19728552,wtg01,NaN,NaN,NaN
8,scada_vestas_train,vestas,2022-10-24 08:30:00,2022-10-24 08:00:00,vestas_wtg01_power_kw10m,power,scada_power_abs_gt_5000,-21934616,wtg01,3659.179,1952.526,NaN
9,scada_vestas_train,vestas,2022-10-27 18:50:00,2022-10-27 18:00:00,vestas_wtg01_power_kw10m,power,scada_power_abs_gt_5000,21934616,wtg01,NaN,NaN,NaN


## 6. Physical Outliers With Train-Label Context

In [7]:
def make_outlier_hourly_context(events: pd.DataFrame, scada_only: bool = False) -> pd.DataFrame:
    if events.empty:
        return pd.DataFrame()
    data = events.copy()
    if scada_only:
        data = data[data["dataset"].str.startswith("scada_")].copy()
    if data.empty:
        return pd.DataFrame()
    context = (
        data.groupby(["dataset", "source", "label_hour"], as_index=False)
        .agg(
            n_event_times=("event_time", "nunique"),
            n_outlier_cells=("column", "size"),
            outlier_columns=("column", unique_join),
            outlier_rules=("rule", unique_join),
            outlier_variable_types=("variable_type", unique_join),
            outlier_turbines=("turbine_id", unique_join),
            min_outlier_value=("value", "min"),
            max_outlier_value=("value", "max"),
        )
        .merge(label_context, on="label_hour", how="left")
        .sort_values(["dataset", "label_hour"])
    )
    return context


outlier_hourly_label_context = make_outlier_hourly_context(outlier_events_detailed, scada_only=False)
scada_outlier_hourly_label_context = make_outlier_hourly_context(outlier_events_detailed, scada_only=True)

outlier_hourly_label_context.to_csv(OUTPUT_PATHS["outlier_hourly_label_context"], index=False, encoding="utf-8-sig")
scada_outlier_hourly_label_context.to_csv(OUTPUT_PATHS["scada_outlier_hourly_label_context"], index=False, encoding="utf-8-sig")

print("All physical outlier hourly label context:", outlier_hourly_label_context.shape)
display(outlier_hourly_label_context.head(30))
print("SCADA physical outlier hourly label context:", scada_outlier_hourly_label_context.shape)
display(scada_outlier_hourly_label_context.head(30))
if not scada_outlier_hourly_label_context.empty:
    print("SCADA outlier context summary by source:")
    display(scada_outlier_hourly_label_context.groupby("source").agg(
        n_hours=("label_hour", "nunique"),
        n_outlier_cells=("n_outlier_cells", "sum"),
        first_hour=("label_hour", "min"),
        last_hour=("label_hour", "max"),
    ).reset_index())

All physical outlier hourly label context: (455, 14)


,dataset,source,label_hour,n_event_times,n_outlier_cells,outlier_columns,outlier_rules,outlier_variable_types,outlier_turbines,min_outlier_value,max_outlier_value,kpx_group_1,kpx_group_2,kpx_group_3
0,scada_vestas_train,vestas,2022-01-11 20:00:00,1,1,vestas_wtg06_power_kw10m,scada_power_abs_gt_5000,power,wtg06,-11765590,-11765590,15056.147,18888.884,NaN
1,scada_vestas_train,vestas,2022-01-12 12:00:00,1,1,vestas_wtg06_power_kw10m,scada_power_abs_gt_5000,power,wtg06,11765837,11765837,7998.126,7217.116,NaN
2,scada_vestas_train,vestas,2022-01-12 13:00:00,1,1,vestas_wtg06_power_kw10m,scada_power_abs_gt_5000,power,wtg06,-11765837,-11765837,6898.926,11252.337,NaN
3,scada_vestas_train,vestas,2022-01-13 10:00:00,2,2,vestas_wtg02_power_kw10m,scada_power_abs_gt_5000,power,wtg02,-13633876,13633875,13580.905,18093.411,NaN
4,scada_vestas_train,vestas,2022-01-16 12:00:00,1,1,vestas_wtg06_power_kw10m,scada_power_abs_gt_5000,power,wtg06,11765834,11765834,12539.558,14521.011,NaN
5,scada_vestas_train,vestas,2022-01-30 19:00:00,1,1,vestas_wtg05_power_kw10m,scada_power_abs_gt_5000,power,wtg05,-13045102,-13045102,5496.000,4541.432,NaN
6,scada_vestas_train,vestas,2022-02-04 18:00:00,2,2,vestas_wtg07_power_kw10m,scada_power_abs_gt_5000,power,wtg07,-9877098,9877091,13450.737,9719.242,NaN
7,scada_vestas_train,vestas,2022-02-17 03:00:00,1,1,vestas_wtg02_power_kw10m,scada_power_abs_gt_5000,power,wtg02,-15012350,-15012350,12163.516,11165.558,NaN
8,scada_vestas_train,vestas,2022-02-17 12:00:00,1,1,vestas_wtg02_power_kw10m,scada_power_abs_gt_5000,power,wtg02,15012345,15012345,535.137,72.316,NaN
9,scada_vestas_train,vestas,2022-02-17 13:00:00,1,1,vestas_wtg02_power_kw10m,scada_power_abs_gt_5000,power,wtg02,-15012339,-15012339,115.705,0.000,NaN


SCADA physical outlier hourly label context: (455, 14)


,dataset,source,label_hour,n_event_times,n_outlier_cells,outlier_columns,outlier_rules,outlier_variable_types,outlier_turbines,min_outlier_value,max_outlier_value,kpx_group_1,kpx_group_2,kpx_group_3
0,scada_vestas_train,vestas,2022-01-11 20:00:00,1,1,vestas_wtg06_power_kw10m,scada_power_abs_gt_5000,power,wtg06,-11765590,-11765590,15056.147,18888.884,NaN
1,scada_vestas_train,vestas,2022-01-12 12:00:00,1,1,vestas_wtg06_power_kw10m,scada_power_abs_gt_5000,power,wtg06,11765837,11765837,7998.126,7217.116,NaN
2,scada_vestas_train,vestas,2022-01-12 13:00:00,1,1,vestas_wtg06_power_kw10m,scada_power_abs_gt_5000,power,wtg06,-11765837,-11765837,6898.926,11252.337,NaN
3,scada_vestas_train,vestas,2022-01-13 10:00:00,2,2,vestas_wtg02_power_kw10m,scada_power_abs_gt_5000,power,wtg02,-13633876,13633875,13580.905,18093.411,NaN
4,scada_vestas_train,vestas,2022-01-16 12:00:00,1,1,vestas_wtg06_power_kw10m,scada_power_abs_gt_5000,power,wtg06,11765834,11765834,12539.558,14521.011,NaN
5,scada_vestas_train,vestas,2022-01-30 19:00:00,1,1,vestas_wtg05_power_kw10m,scada_power_abs_gt_5000,power,wtg05,-13045102,-13045102,5496.000,4541.432,NaN
6,scada_vestas_train,vestas,2022-02-04 18:00:00,2,2,vestas_wtg07_power_kw10m,scada_power_abs_gt_5000,power,wtg07,-9877098,9877091,13450.737,9719.242,NaN
7,scada_vestas_train,vestas,2022-02-17 03:00:00,1,1,vestas_wtg02_power_kw10m,scada_power_abs_gt_5000,power,wtg02,-15012350,-15012350,12163.516,11165.558,NaN
8,scada_vestas_train,vestas,2022-02-17 12:00:00,1,1,vestas_wtg02_power_kw10m,scada_power_abs_gt_5000,power,wtg02,15012345,15012345,535.137,72.316,NaN
9,scada_vestas_train,vestas,2022-02-17 13:00:00,1,1,vestas_wtg02_power_kw10m,scada_power_abs_gt_5000,power,wtg02,-15012339,-15012339,115.705,0.000,NaN


SCADA outlier context summary by source:


,source,n_hours,n_outlier_cells,first_hour,last_hour
0,vestas,455,868,2022-01-11 20:00:00,2024-12-28 11:00:00


## 7. Negative Generation Audit

This section records every negative generation value in train labels and SCADA power columns. This is separate from the strict physical-outlier audit because small negative SCADA power can be common enough that it is useful to inspect it independently.


In [8]:
def generation_columns_for_dataset(dataset_name: str, df: pd.DataFrame) -> list[str]:
    if dataset_name == "train_labels":
        return [col for col in LABEL_COLS if col in df.columns]
    if dataset_name.startswith("scada_"):
        return [col for col in df.columns if re.search(r"_power_kw10m$", col)]
    return []


def negative_generation_events_for_dataset(dataset_name: str, df: pd.DataFrame) -> pd.DataFrame:
    source = dataset_source(dataset_name)
    time_col = TIME_COLUMNS[dataset_name]
    event_frames = []
    for col in generation_columns_for_dataset(dataset_name, df):
        values = df[col]
        mask = values.notna() & (values < 0)
        if not mask.any():
            continue
        frame = df.loc[mask, [time_col, col]].copy()
        frame["dataset"] = dataset_name
        frame["source"] = source
        frame["event_time"] = frame[time_col]
        frame["label_hour"] = label_join_hour(dataset_name, frame["event_time"])
        frame["column"] = col
        frame["event_type"] = "negative_generation"
        frame["variable_type"] = "label_generation" if dataset_name == "train_labels" else "scada_power"
        frame["value"] = frame[col]
        frame["turbine_id"] = turbine_id_from_scada_column(col) if dataset_name.startswith("scada_") else np.nan
        event_frames.append(frame.drop(columns=[time_col, col]))
    if not event_frames:
        return pd.DataFrame(columns=[
            "dataset", "source", "event_time", "label_hour", "column", "event_type",
            "variable_type", "value", "turbine_id",
        ])
    return pd.concat(event_frames, ignore_index=True)


negative_event_frames = [negative_generation_events_for_dataset(name, df) for name, df in datasets.items()]
negative_event_frames = [frame for frame in negative_event_frames if not frame.empty]
if negative_event_frames:
    negative_generation_events_detailed = pd.concat(negative_event_frames, ignore_index=True)
    negative_generation_events_detailed["event_time"] = pd.to_datetime(negative_generation_events_detailed["event_time"])
    negative_generation_events_detailed["label_hour"] = pd.to_datetime(negative_generation_events_detailed["label_hour"])
    negative_generation_events_detailed = negative_generation_events_detailed.merge(label_context, on="label_hour", how="left")
else:
    negative_generation_events_detailed = pd.DataFrame(columns=[
        "dataset", "source", "event_time", "label_hour", "column", "event_type",
        "variable_type", "value", "turbine_id", *LABEL_COLS,
    ])

def summarize_negative_generation_columns() -> pd.DataFrame:
    rows = []
    for dataset_name, df in datasets.items():
        source = dataset_source(dataset_name)
        time_col = TIME_COLUMNS[dataset_name]
        for col in generation_columns_for_dataset(dataset_name, df):
            values = df[col]
            mask = values.notna() & (values < 0)
            event_times = pd.to_datetime(df.loc[mask, time_col]) if mask.any() else pd.Series([], dtype="datetime64[ns]")
            rows.append({
                "dataset": dataset_name,
                "source": source,
                "column": col,
                "variable_type": "label_generation" if dataset_name == "train_labels" else "scada_power",
                "negative_count": int(mask.sum()),
                "first_event_time": event_times.min() if mask.any() else pd.NaT,
                "last_event_time": event_times.max() if mask.any() else pd.NaT,
                "min_value": values.loc[mask].min() if mask.any() else np.nan,
                "max_value": values.loc[mask].max() if mask.any() else np.nan,
            })
    return pd.DataFrame(rows).sort_values(["negative_count", "dataset", "column"], ascending=[False, True, True]).reset_index(drop=True)


def summarize_negative_generation_datasets(column_summary: pd.DataFrame, events: pd.DataFrame) -> pd.DataFrame:
    base = (
        column_summary.groupby(["dataset", "source"], as_index=False)
        .agg(
            n_negative_cells=("negative_count", "sum"),
            n_columns_checked=("column", "nunique"),
            n_columns_with_negatives=("negative_count", lambda s: int((s > 0).sum())),
            first_event_time=("first_event_time", "min"),
            last_event_time=("last_event_time", "max"),
        )
    )
    if events.empty:
        base["n_event_hours"] = 0
    else:
        hours = events.groupby(["dataset", "source"], as_index=False).agg(n_event_hours=("label_hour", "nunique"))
        base = base.merge(hours, on=["dataset", "source"], how="left")
        base["n_event_hours"] = base["n_event_hours"].fillna(0).astype(int)
    return base[[
        "dataset", "source", "n_negative_cells", "n_event_hours", "n_columns_checked",
        "n_columns_with_negatives", "first_event_time", "last_event_time",
    ]].sort_values(["n_negative_cells", "dataset"], ascending=[False, True]).reset_index(drop=True)


negative_generation_column_summary = summarize_negative_generation_columns()
negative_generation_dataset_summary = summarize_negative_generation_datasets(
    negative_generation_column_summary,
    negative_generation_events_detailed,
)


def make_negative_generation_hourly_context(events: pd.DataFrame, scada_only: bool = False) -> pd.DataFrame:
    if events.empty:
        return pd.DataFrame(columns=[
            "dataset", "source", "label_hour", "n_event_times", "n_negative_cells",
            "negative_columns", "negative_variable_types", "negative_turbines",
            "min_negative_value", "max_negative_value", *LABEL_COLS,
        ])
    data = events.copy()
    if scada_only:
        data = data[data["dataset"].str.startswith("scada_")].copy()
    if data.empty:
        return pd.DataFrame(columns=[
            "dataset", "source", "label_hour", "n_event_times", "n_negative_cells",
            "negative_columns", "negative_variable_types", "negative_turbines",
            "min_negative_value", "max_negative_value", *LABEL_COLS,
        ])
    context = (
        data.groupby(["dataset", "source", "label_hour"], as_index=False)
        .agg(
            n_event_times=("event_time", "nunique"),
            n_negative_cells=("column", "size"),
            negative_columns=("column", unique_join),
            negative_variable_types=("variable_type", unique_join),
            negative_turbines=("turbine_id", unique_join),
            min_negative_value=("value", "min"),
            max_negative_value=("value", "max"),
        )
        .merge(label_context, on="label_hour", how="left")
        .sort_values(["dataset", "label_hour"])
    )
    return context


negative_generation_hourly_label_context = make_negative_generation_hourly_context(negative_generation_events_detailed, scada_only=False)
scada_negative_generation_hourly_label_context = make_negative_generation_hourly_context(negative_generation_events_detailed, scada_only=True)

negative_generation_dataset_summary.to_csv(OUTPUT_PATHS["negative_generation_dataset_summary"], index=False, encoding="utf-8-sig")
negative_generation_column_summary.to_csv(OUTPUT_PATHS["negative_generation_column_summary"], index=False, encoding="utf-8-sig")
negative_generation_events_detailed.to_csv(OUTPUT_PATHS["negative_generation_events_detailed"], index=False, encoding="utf-8-sig")
negative_generation_hourly_label_context.to_csv(OUTPUT_PATHS["negative_generation_hourly_label_context"], index=False, encoding="utf-8-sig")
scada_negative_generation_hourly_label_context.to_csv(OUTPUT_PATHS["scada_negative_generation_hourly_label_context"], index=False, encoding="utf-8-sig")

print("Negative generation dataset summary:")
display(negative_generation_dataset_summary)
print("Negative generation column summary:")
display(negative_generation_column_summary)
print("Negative generation detailed events shape:", negative_generation_events_detailed.shape)
display(negative_generation_events_detailed.head(20))
print("SCADA negative generation hourly label context:", scada_negative_generation_hourly_label_context.shape)
display(scada_negative_generation_hourly_label_context.head(20))


Negative generation dataset summary:


,dataset,source,n_negative_cells,n_event_hours,n_columns_checked,n_columns_with_negatives,first_event_time,last_event_time
0,scada_vestas_train,vestas,317221,15799,12,12,2022-01-01 04:40:00,2024-12-31 14:00:00
1,scada_unison_train,unison,0,0,5,0,NaT,NaT
2,train_labels,label,0,0,3,0,NaT,NaT


Negative generation column summary:


,dataset,source,column,variable_type,negative_count,first_event_time,last_event_time,min_value,max_value
0,scada_vestas_train,vestas,vestas_wtg07_power_kw10m,scada_power,37209,2022-01-01 04:40:00,2024-12-31 13:10:00,-29770834.0,-1.0
1,scada_vestas_train,vestas,vestas_wtg06_power_kw10m,scada_power,28489,2022-01-01 11:30:00,2024-12-24 14:20:00,-34184409.0,-1.0
2,scada_vestas_train,vestas,vestas_wtg08_power_kw10m,scada_power,28198,2022-01-01 12:20:00,2024-12-26 12:40:00,-42301001.0,-1.0
3,scada_vestas_train,vestas,vestas_wtg12_power_kw10m,scada_power,28135,2022-01-04 21:20:00,2024-12-24 14:10:00,-41447795.0,-1.0
4,scada_vestas_train,vestas,vestas_wtg09_power_kw10m,scada_power,27372,2022-01-01 12:00:00,2024-12-24 15:40:00,-43556999.0,-1.0
5,scada_vestas_train,vestas,vestas_wtg05_power_kw10m,scada_power,27255,2022-01-01 12:10:00,2024-12-31 14:00:00,-33717814.0,-1.0
6,scada_vestas_train,vestas,vestas_wtg04_power_kw10m,scada_power,26723,2022-01-01 12:20:00,2024-12-24 13:40:00,-40642699.0,-1.0
7,scada_vestas_train,vestas,vestas_wtg01_power_kw10m,scada_power,24784,2022-01-04 17:00:00,2024-12-28 13:30:00,-42512957.0,-1.0
8,scada_vestas_train,vestas,vestas_wtg02_power_kw10m,scada_power,23751,2022-01-01 12:20:00,2024-12-26 05:40:00,-41959813.0,-1.0
9,scada_vestas_train,vestas,vestas_wtg11_power_kw10m,scada_power,23658,2022-01-01 12:10:00,2024-12-28 15:30:00,-51770425.0,-1.0


Negative generation detailed events shape: (317221, 12)


,dataset,source,event_time,label_hour,column,event_type,variable_type,value,turbine_id,kpx_group_1,kpx_group_2,kpx_group_3
0,scada_vestas_train,vestas,2022-01-04 17:00:00,2022-01-04 17:00:00,vestas_wtg01_power_kw10m,negative_generation,scada_power,-1,wtg01,448.358,911.179,NaN
1,scada_vestas_train,vestas,2022-01-04 21:10:00,2022-01-04 21:00:00,vestas_wtg01_power_kw10m,negative_generation,scada_power,-1,wtg01,983.495,809.937,NaN
2,scada_vestas_train,vestas,2022-01-04 21:20:00,2022-01-04 21:00:00,vestas_wtg01_power_kw10m,negative_generation,scada_power,-4,wtg01,983.495,809.937,NaN
3,scada_vestas_train,vestas,2022-01-04 21:30:00,2022-01-04 21:00:00,vestas_wtg01_power_kw10m,negative_generation,scada_power,-5,wtg01,983.495,809.937,NaN
4,scada_vestas_train,vestas,2022-01-04 21:40:00,2022-01-04 21:00:00,vestas_wtg01_power_kw10m,negative_generation,scada_power,-4,wtg01,983.495,809.937,NaN
5,scada_vestas_train,vestas,2022-01-04 21:50:00,2022-01-04 21:00:00,vestas_wtg01_power_kw10m,negative_generation,scada_power,-4,wtg01,983.495,809.937,NaN
6,scada_vestas_train,vestas,2022-01-04 22:00:00,2022-01-04 22:00:00,vestas_wtg01_power_kw10m,negative_generation,scada_power,-3,wtg01,0.000,0.000,NaN
7,scada_vestas_train,vestas,2022-01-04 22:10:00,2022-01-04 22:00:00,vestas_wtg01_power_kw10m,negative_generation,scada_power,-5,wtg01,0.000,0.000,NaN
8,scada_vestas_train,vestas,2022-01-04 22:20:00,2022-01-04 22:00:00,vestas_wtg01_power_kw10m,negative_generation,scada_power,-3,wtg01,0.000,0.000,NaN
9,scada_vestas_train,vestas,2022-01-04 22:30:00,2022-01-04 22:00:00,vestas_wtg01_power_kw10m,negative_generation,scada_power,-5,wtg01,0.000,0.000,NaN


SCADA negative generation hourly label context: (15799, 13)


,dataset,source,label_hour,n_event_times,n_negative_cells,negative_columns,negative_variable_types,negative_turbines,min_negative_value,max_negative_value,kpx_group_1,kpx_group_2,kpx_group_3
0,scada_vestas_train,vestas,2022-01-01 04:00:00,2,2,vestas_wtg07_power_kw10m,scada_power,wtg07,-5,-4,17167.768,13841.242,NaN
1,scada_vestas_train,vestas,2022-01-01 05:00:00,3,3,vestas_wtg07_power_kw10m,scada_power,wtg07,-4,-4,19134.758,14014.800,NaN
2,scada_vestas_train,vestas,2022-01-01 08:00:00,1,1,vestas_wtg07_power_kw10m,scada_power,wtg07,-2,-2,9111.789,5872.042,NaN
3,scada_vestas_train,vestas,2022-01-01 09:00:00,2,2,vestas_wtg07_power_kw10m,scada_power,wtg07,-4,-4,3688.105,2183.937,NaN
4,scada_vestas_train,vestas,2022-01-01 10:00:00,4,4,vestas_wtg07_power_kw10m,scada_power,wtg07,-5,-1,3803.811,1851.284,NaN
5,scada_vestas_train,vestas,2022-01-01 11:00:00,2,2,vestas_wtg06_power_kw10m|vestas_wtg07_power_kw10m,scada_power,wtg06|wtg07,-5,-3,4020.758,1345.074,NaN
6,scada_vestas_train,vestas,2022-01-01 12:00:00,6,27,vestas_wtg02_power_kw10m|vestas_wtg04_power_kw10m|vestas_wtg05_power_kw10m|vestas_wtg06_power_kw10m|vestas_wtg07_power_kw10m|vestas_wtg08_power_kw10m|vestas_wtg09_power_kw10m|vestas_wtg10_power_kw10m|vestas_wtg11_power_kw10m,scada_power,wtg02|wtg04|wtg05|wtg06|wtg07|wtg08|wtg09|wtg10|wtg11,-6,-1,940.105,404.968,NaN
7,scada_vestas_train,vestas,2022-01-01 15:00:00,1,1,vestas_wtg04_power_kw10m,scada_power,wtg04,-1,-1,2661.221,3774.884,NaN
8,scada_vestas_train,vestas,2022-01-01 16:00:00,4,4,vestas_wtg07_power_kw10m,scada_power,wtg07,-4,-2,997.958,3181.895,NaN
9,scada_vestas_train,vestas,2022-01-01 19:00:00,2,2,vestas_wtg07_power_kw10m,scada_power,wtg07,-5,-4,3051.726,2097.158,NaN


## 8. Label Capacity Warnings

This table is separate from the strict physical-outlier table. It records values above nominal capacity but below the 105% physical-outlier threshold.

In [9]:
warning_frames = []
for col, capacity in LABEL_CAPACITY_KWH.items():
    mask = labels[col].notna() & (labels[col] > capacity) & (labels[col] <= capacity * LABEL_CAPACITY_TOLERANCE)
    if mask.any():
        frame = labels.loc[mask, ["kst_dtm"] + LABEL_COLS].copy()
        frame["target"] = col
        frame["capacity_kwh"] = capacity
        frame["value"] = frame[col]
        frame["excess_kwh"] = frame["value"] - capacity
        frame["excess_rate"] = frame["value"] / capacity - 1
        warning_frames.append(frame)
label_capacity_warning_events = pd.concat(warning_frames, ignore_index=True) if warning_frames else pd.DataFrame()
label_capacity_warning_events.to_csv(OUTPUT_PATHS["label_capacity_warning_events"], index=False, encoding="utf-8-sig")
print("Label capacity warning events:", label_capacity_warning_events.shape)
display(label_capacity_warning_events.head(50))

Label capacity warning events: (38, 9)


,kst_dtm,kpx_group_1,kpx_group_2,kpx_group_3,target,capacity_kwh,value,excess_kwh,excess_rate
0,2023-01-11 06:00:00,19698.821,21014.968,21058.358,kpx_group_3,21000.0,21058.358,58.358,0.002779
1,2023-02-28 02:00:00,20725.705,20436.442,21087.284,kpx_group_3,21000.0,21087.284,87.284,0.004156
2,2023-04-13 00:00:00,20826.947,20393.053,21058.358,kpx_group_3,21000.0,21058.358,58.358,0.002779
3,2023-04-13 02:00:00,20783.558,20335.200,21058.358,kpx_group_3,21000.0,21058.358,58.358,0.002779
4,2023-04-13 06:00:00,21246.379,20436.442,21000.505,kpx_group_3,21000.0,21000.505,0.505,0.000024
5,2023-07-14 18:00:00,18426.063,18527.305,21072.821,kpx_group_3,21000.0,21072.821,72.821,0.003468
6,2023-11-18 21:00:00,19322.779,20320.737,21043.895,kpx_group_3,21000.0,21043.895,43.895,0.002090
7,2023-11-21 21:00:00,21246.379,18570.695,21072.821,kpx_group_3,21000.0,21072.821,72.821,0.003468
8,2023-12-25 01:00:00,16994.211,20233.958,21000.505,kpx_group_3,21000.0,21000.505,0.505,0.000024
9,2023-12-26 01:00:00,20378.589,21188.526,21000.505,kpx_group_3,21000.0,21000.505,0.505,0.000024


## 9. Final Output Summary

In [10]:
print("Output files:")
for name, path in OUTPUT_PATHS.items():
    print(f"- {name}: {path}")

print("Expected missing exclusions:")
display(expected_missing_exclusions)

print("Missing dataset summary, after expected exclusions:")
display(missing_dataset_summary)

print("Physical outlier dataset summary:")
display(outlier_dataset_summary)

print("Negative generation dataset summary:")
display(negative_generation_dataset_summary)

print("SCADA missing hourly label context sample:")
display(scada_missing_hourly_label_context.head(20))

print("SCADA physical outlier hourly label context sample:")
display(scada_outlier_hourly_label_context.head(20))

print("SCADA negative generation hourly label context sample:")
display(scada_negative_generation_hourly_label_context.head(20))


Output files:
- missing_dataset_summary: C:\Users\심현석\Documents\BARAM 2026\outputs\missing_outlier_audit\missing_dataset_summary.csv
- missing_column_summary: C:\Users\심현석\Documents\BARAM 2026\outputs\missing_outlier_audit\missing_column_summary.csv
- missing_events_detailed: C:\Users\심현석\Documents\BARAM 2026\outputs\missing_outlier_audit\missing_events_detailed.csv
- expected_missing_exclusions: C:\Users\심현석\Documents\BARAM 2026\outputs\missing_outlier_audit\expected_missing_exclusions.csv
- scada_missing_hourly_label_context: C:\Users\심현석\Documents\BARAM 2026\outputs\missing_outlier_audit\scada_missing_hourly_label_context.csv
- outlier_rule_config: C:\Users\심현석\Documents\BARAM 2026\outputs\missing_outlier_audit\physical_outlier_rule_config.csv
- outlier_dataset_summary: C:\Users\심현석\Documents\BARAM 2026\outputs\missing_outlier_audit\physical_outlier_dataset_summary.csv
- outlier_column_summary: C:\Users\심현석\Documents\BARAM 2026\outputs\missing_outlier_audit\physical_outlier_column_s

,dataset,source,column,excluded_missing_count,first_excluded_time,last_excluded_time,reason
0,train_labels,label,kpx_group_3,8760,2022-01-01 01:00:00,2023-01-01,Expected missing values through 2023-01-01 00:00:00


Missing dataset summary, after expected exclusions:


,dataset,source,n_rows,n_columns,n_missing_cells,n_rows_with_missing,n_columns_with_missing,missing_cell_rate
0,train_labels,label,26304,4,213,106,3,0.002024
1,ldaps_train,ldaps,420864,35,0,0,0,0.000000
2,gfs_train,gfs,236736,40,0,0,0,0.000000
3,scada_vestas_train,vestas,157819,37,0,0,0,0.000000
4,scada_unison_train,unison,105264,16,9511,4734,15,0.005647


Physical outlier dataset summary:


...

,dataset,source,label_hour,n_event_rows,n_missing_cells,missing_columns,missing_turbines,missing_variable_types,kpx_group_1,kpx_group_2,kpx_group_3
0,scada_unison_train,unison,2023-01-06 09:00:00,3,3,unison_wtg04_ws,wtg04,ws,9343.200,12973.453,7217.116
1,scada_unison_train,unison,2023-01-11 09:00:00,1,1,unison_wtg05_ws,wtg05,ws,19843.453,20754.632,14159.432
2,scada_unison_train,unison,2023-01-12 09:00:00,1,1,unison_wtg05_ws,wtg05,ws,17095.453,12351.537,3413.305
3,scada_unison_train,unison,2023-01-13 09:00:00,2,2,unison_wtg03_ws|unison_wtg05_ws,wtg03|wtg05,ws,10095.284,10601.495,4454.653
4,scada_unison_train,unison,2023-01-13 10:00:00,5,6,unison_wtg03_ws|unison_wtg05_ws,wtg03|wtg05,ws,11686.232,9863.874,3688.105
5,scada_unison_train,unison,2023-01-13 11:00:00,3,3,unison_wtg05_ws,wtg05,ws,11020.926,12438.316,5293.516
6,scada_unison_train,unison,2023-01-18 10:00:00,4,12,unison_wtg05_power_kw10m|unison_wtg05_ws|unison_wtg05_wd,wtg05,power_kw10m|ws|wd,1836.821,2256.253,592.989
7,scada_unison_train,unison,2023-01-18 11:00:00,6,18,unison_wtg05_power_kw10m|unison_wtg05_ws|unison_wtg05_wd,wtg05,power_kw10m|ws|wd,882.253,1084.737,216.947
8,scada_unison_train,unison,2023-01-18 12:00:00,1,2,unison_wtg05_power_kw10m|unison_wtg05_ws,wtg05,power_kw10m|ws,650.842,433.895,86.779
9,scada_unison_train,unison,2023-01-18 16:00:00,1,1,unison_wtg04_ws,wtg04,ws,7231.579,3601.326,2863.705


SCADA physical outlier hourly label context sample:


,dataset,source,label_hour,n_event_times,n_outlier_cells,outlier_columns,outlier_rules,outlier_variable_types,outlier_turbines,min_outlier_value,max_outlier_value,kpx_group_1,kpx_group_2,kpx_group_3
0,scada_vestas_train,vestas,2022-01-11 20:00:00,1,1,vestas_wtg06_power_kw10m,scada_power_abs_gt_5000,power,wtg06,-11765590,-11765590,15056.147,18888.884,NaN
1,scada_vestas_train,vestas,2022-01-12 12:00:00,1,1,vestas_wtg06_power_kw10m,scada_power_abs_gt_5000,power,wtg06,11765837,11765837,7998.126,7217.116,NaN
2,scada_vestas_train,vestas,2022-01-12 13:00:00,1,1,vestas_wtg06_power_kw10m,scada_power_abs_gt_5000,power,wtg06,-11765837,-11765837,6898.926,11252.337,NaN
3,scada_vestas_train,vestas,2022-01-13 10:00:00,2,2,vestas_wtg02_power_kw10m,scada_power_abs_gt_5000,power,wtg02,-13633876,13633875,13580.905,18093.411,NaN
4,scada_vestas_train,vestas,2022-01-16 12:00:00,1,1,vestas_wtg06_power_kw10m,scada_power_abs_gt_5000,power,wtg06,11765834,11765834,12539.558,14521.011,NaN
5,scada_vestas_train,vestas,2022-01-30 19:00:00,1,1,vestas_wtg05_power_kw10m,scada_power_abs_gt_5000,power,wtg05,-13045102,-13045102,5496.000,4541.432,NaN
6,scada_vestas_train,vestas,2022-02-04 18:00:00,2,2,vestas_wtg07_power_kw10m,scada_power_abs_gt_5000,power,wtg07,-9877098,9877091,13450.737,9719.242,NaN
7,scada_vestas_train,vestas,2022-02-17 03:00:00,1,1,vestas_wtg02_power_kw10m,scada_power_abs_gt_5000,power,wtg02,-15012350,-15012350,12163.516,11165.558,NaN
8,scada_vestas_train,vestas,2022-02-17 12:00:00,1,1,vestas_wtg02_power_kw10m,scada_power_abs_gt_5000,power,wtg02,15012345,15012345,535.137,72.316,NaN
9,scada_vestas_train,vestas,2022-02-17 13:00:00,1,1,vestas_wtg02_power_kw10m,scada_power_abs_gt_5000,power,wtg02,-15012339,-15012339,115.705,0.000,NaN


SCADA negative generation hourly label context sample:


,dataset,source,label_hour,n_event_times,n_negative_cells,negative_columns,negative_variable_types,negative_turbines,min_negative_value,max_negative_value,kpx_group_1,kpx_group_2,kpx_group_3
0,scada_vestas_train,vestas,2022-01-01 04:00:00,2,2,vestas_wtg07_power_kw10m,scada_power,wtg07,-5,-4,17167.768,13841.242,NaN
1,scada_vestas_train,vestas,2022-01-01 05:00:00,3,3,vestas_wtg07_power_kw10m,scada_power,wtg07,-4,-4,19134.758,14014.800,NaN
2,scada_vestas_train,vestas,2022-01-01 08:00:00,1,1,vestas_wtg07_power_kw10m,scada_power,wtg07,-2,-2,9111.789,5872.042,NaN
3,scada_vestas_train,vestas,2022-01-01 09:00:00,2,2,vestas_wtg07_power_kw10m,scada_power,wtg07,-4,-4,3688.105,2183.937,NaN
4,scada_vestas_train,vestas,2022-01-01 10:00:00,4,4,vestas_wtg07_power_kw10m,scada_power,wtg07,-5,-1,3803.811,1851.284,NaN
5,scada_vestas_train,vestas,2022-01-01 11:00:00,2,2,vestas_wtg06_power_kw10m|vestas_wtg07_power_kw10m,scada_power,wtg06|wtg07,-5,-3,4020.758,1345.074,NaN
6,scada_vestas_train,vestas,2022-01-01 12:00:00,6,27,vestas_wtg02_power_kw10m|vestas_wtg04_power_kw10m|vestas_wtg05_power_kw10m|vestas_wtg06_power_kw10m|vestas_wtg07_power_kw10m|vestas_wtg08_power_kw10m|vestas_wtg09_power_kw10m|vestas_wtg10_power_kw10m|vestas_wtg11_power_kw10m,scada_power,wtg02|wtg04|wtg05|wtg06|wtg07|wtg08|wtg09|wtg10|wtg11,-6,-1,940.105,404.968,NaN
7,scada_vestas_train,vestas,2022-01-01 15:00:00,1,1,vestas_wtg04_power_kw10m,scada_power,wtg04,-1,-1,2661.221,3774.884,NaN
8,scada_vestas_train,vestas,2022-01-01 16:00:00,4,4,vestas_wtg07_power_kw10m,scada_power,wtg07,-4,-2,997.958,3181.895,NaN
9,scada_vestas_train,vestas,2022-01-01 19:00:00,2,2,vestas_wtg07_power_kw10m,scada_power,wtg07,-5,-4,3051.726,2097.158,NaN
